In [1]:
import pickle
import matminer
import numpy as np
import pandas as pd
from matminer.datasets import load_dataset
from pymatgen.core.composition import Composition
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from matminer.featurizers.composition import ElementProperty, ElementFraction
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, f1_score, precision_score, recall_score

In [2]:
# Load model
with open('sc_efep_best_model_yet.pkl', 'rb') as file:
    model_efep = pickle.load(file)

In [3]:
# Split the dataframe into features and target

def featurize(data):
    data = data.dropna()
    ep_featurizer = ElementProperty.from_preset('magpie')
    ep_ftd = ep_featurizer.featurize_dataframe(data, col_id='_Composition', ignore_errors=True)
    
    # Use ElementFraction instead of ElementProperty
    ef_featurizer = ElementFraction()
    ef_ftd = ef_featurizer.featurize_dataframe(data, col_id='_Composition', ignore_errors=True)
    return ep_ftd, ef_ftd

In [4]:
ef_ftd = pd.read_csv("df_sc_ef_ElementFraction_ftd.csv").dropna().reset_index()
ep_ftd = pd.read_csv("df_sc_ep_ElementProperty_Magpie_ftd.csv").dropna().reset_index()

ef_ftd = ef_ftd.drop_duplicates(subset=["Critical Temp", "_Composition"])
ep_ftd = ep_ftd.drop_duplicates(subset=["Critical Temp", "_Composition"])

ef_ftd = ef_ftd[ef_ftd["Critical Temp"] >= 10]
ep_ftd = ep_ftd[ep_ftd["Critical Temp"] >= 10]

In [5]:
ep_ftd

,index,composition,Critical Temp,_Composition,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,...,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,0,Ba0.4K0.6Fe2As2,31.20,Ba0.4 K0.6 Fe2 As2,19.0,56.0,37.0,30.360000,6.214400,26.0,...,2.110663,0.844265,1.013118,0.0,166.0,229.0,63.0,203.800000,30.240000,166.0
1,1,Ca0.4Ba1.25La1.25Cu3O6.98,40.10,Ca0.4 Ba1.25 La1.25 Cu3 O6.98,8.0,57.0,49.0,22.677795,16.074864,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,229.0,217.0,106.949534,102.911141,12.0
6,6,La1.71Sr0.29Cu0.94Co0.06O4,33.00,La1.71 Sr0.29 Cu0.94 Co0.06 O4,8.0,57.0,49.0,24.195714,18.509388,8.0,...,1.548471,0.013273,0.026318,0.0,12.0,225.0,213.0,95.447143,95.368163,12.0
10,10,Nb3Sn0.85Tl0.15,18.20,Nb3 Sn0.85 Tl0.15,41.0,81.0,40.0,44.412500,5.118750,41.0,...,0.000000,0.000000,0.000000,0.0,141.0,229.0,88.0,208.987500,30.018750,229.0
11,11,Pb0.5Cu0.5Sr0.9La1.1Cu1O5.16,28.10,Pb0.5 Cu1.5 Sr0.9 La1.1 O5.16,8.0,82.0,74.0,24.310044,18.375508,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,101.290393,100.597910,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16392,16392,Dy1Ba2Cu2.8Zn0.2O6.95,13.00,Dy1 Ba2 Cu2.8 Zn0.2 O6.95,8.0,66.0,58.0,24.772201,18.002594,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,229.0,217.0,108.432432,103.506626,12.0
16393,16393,Bi2Ca2.5Sm0.5Cu2O8.33,22.00,Bi2 Ca2.5 Sm0.5 Cu2 O8.33,8.0,83.0,75.0,24.242661,19.035620,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,79.547293,91.032425,12.0
16400,16400,La1.78Sr0.22Cu0.9975Zn0.0025O4,19.25,La1.78 Sr0.22 Cu0.9975 Zn0.0025 O4,8.0,57.0,49.0,24.403214,18.746531,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,95.391786,95.304898,12.0
16403,16403,Pb2Sr2Ho0.5Ca0.5Cu2.982Al0.018O8,63.60,Pb2 Sr2 Ho0.5 Ca0.5 Cu2.982 Al0.018 O8,8.0,82.0,74.0,27.138250,19.616202,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,117.531250,105.531250,12.0


In [6]:
ep_ftd

,index,composition,Critical Temp,_Composition,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,...,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,0,Ba0.4K0.6Fe2As2,31.20,Ba0.4 K0.6 Fe2 As2,19.0,56.0,37.0,30.360000,6.214400,26.0,...,2.110663,0.844265,1.013118,0.0,166.0,229.0,63.0,203.800000,30.240000,166.0
1,1,Ca0.4Ba1.25La1.25Cu3O6.98,40.10,Ca0.4 Ba1.25 La1.25 Cu3 O6.98,8.0,57.0,49.0,22.677795,16.074864,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,229.0,217.0,106.949534,102.911141,12.0
6,6,La1.71Sr0.29Cu0.94Co0.06O4,33.00,La1.71 Sr0.29 Cu0.94 Co0.06 O4,8.0,57.0,49.0,24.195714,18.509388,8.0,...,1.548471,0.013273,0.026318,0.0,12.0,225.0,213.0,95.447143,95.368163,12.0
10,10,Nb3Sn0.85Tl0.15,18.20,Nb3 Sn0.85 Tl0.15,41.0,81.0,40.0,44.412500,5.118750,41.0,...,0.000000,0.000000,0.000000,0.0,141.0,229.0,88.0,208.987500,30.018750,229.0
11,11,Pb0.5Cu0.5Sr0.9La1.1Cu1O5.16,28.10,Pb0.5 Cu1.5 Sr0.9 La1.1 O5.16,8.0,82.0,74.0,24.310044,18.375508,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,101.290393,100.597910,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16392,16392,Dy1Ba2Cu2.8Zn0.2O6.95,13.00,Dy1 Ba2 Cu2.8 Zn0.2 O6.95,8.0,66.0,58.0,24.772201,18.002594,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,229.0,217.0,108.432432,103.506626,12.0
16393,16393,Bi2Ca2.5Sm0.5Cu2O8.33,22.00,Bi2 Ca2.5 Sm0.5 Cu2 O8.33,8.0,83.0,75.0,24.242661,19.035620,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,79.547293,91.032425,12.0
16400,16400,La1.78Sr0.22Cu0.9975Zn0.0025O4,19.25,La1.78 Sr0.22 Cu0.9975 Zn0.0025 O4,8.0,57.0,49.0,24.403214,18.746531,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,95.391786,95.304898,12.0
16403,16403,Pb2Sr2Ho0.5Ca0.5Cu2.982Al0.018O8,63.60,Pb2 Sr2 Ho0.5 Ca0.5 Cu2.982 Al0.018 O8,8.0,82.0,74.0,27.138250,19.616202,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,117.531250,105.531250,12.0


In [7]:
def create_composition(formula):
    data = pd.DataFrame({"composition": [formula]})
    try:
        data["_Composition"] = [Composition(formula)]
        return data
    except ValueError:
        print(f"Error parsing formula: {formula}")

In [8]:
def record_indices(ep_ftd, ef_ftd):
    formula = input("Enter Element Composition: ")
    # Check if composition exists in the original dataset
    data = create_composition(formula)
    print(data)
    composition = data["composition"].tolist()[0]

    if composition in ep_ftd['composition'].tolist():
        final_ep_ftd = ep_ftd[ep_ftd['composition'] == composition]
        final_ef_ftd = ef_ftd[ep_ftd['composition'] == composition]
        print("\nIs there any null value?", final_ep_ftd.isnull().any(axis = 1))
        return final_ep_ftd, final_ef_ftd
    else:
        final_ep_ftd, final_ef_ftd = featurize(data)
        print("\nIs there any null value?", final_ep_ftd.isnull().any(axis = 1))
        return final_ep_ftd, final_ef_ftd

final_ep_ftd, final_ef_ftd = record_indices(ep_ftd, ef_ftd)

Enter Element Composition:  Bi2Ca2.5Sm0.5Cu2O8.33


             composition         _Composition
0  Bi2Ca2.5Sm0.5Cu2O8.33  (Bi, Ca, Sm, Cu, O)

Is there any null value? 16393    False
dtype: bool


In [9]:
def preprocess_for_model2(ep_ftd, ef_ftd):
    ef_ftd = ef_ftd.iloc[:, 2:]
    ep_ftd = ep_ftd.iloc[:, 2:]
    
    print(ef_ftd.shape)
    print(ep_ftd.shape)
    
    merged_df = pd.merge(ef_ftd, ep_ftd, left_on=["Critical Temp", "_Composition"], right_on=["Critical Temp", "_Composition"], how="inner")
    merged_df = pd.merge(ef_ftd, ep_ftd, left_on=["Critical Temp", "_Composition"], right_on=["Critical Temp", "_Composition"], how="inner")

    efep_X = merged_df.iloc[:, 2:]
    efep_y = merged_df['Critical Temp']    
    
    print(efep_X.shape)
    return efep_X, efep_y, merged_df

In [10]:
efep_X, efep_y, merged_df = preprocess_for_model2(final_ep_ftd, final_ef_ftd)

pred_efep = model_efep.predict(efep_X)

(1, 105)
(1, 134)
(1, 235)


In [11]:
print("Actual:", efep_y[0])
print("Predicted:", pred_efep[0])

Actual: 22.0
Predicted: 38.93974071429405
